# Feature audit companion

## tl;dr

All 130 columns were profiled without modifying canonical data. Core position/time/velocity fields have no nulls; SST, uncertainty and hardware require field-specific handling. Recommendations remain proposals. The main human-readable audit is `docs/feature_quality_audit_20261016.md`.

## Context & Methods

### Key assumptions

Recommendations use development data. R rows are observed origins, not horizon-qualified pairs. Metadata are counted per ID. Nulls, declared fills, finite QC failures and not-applicable evidence are distinct. These cells review saved audit results; the exact full census is in the retained audit script. They were executed sequentially as standard Python, without a Jupyter kernel.

In [1]:
from pathlib import Path
import json
import pandas as pd
roots = [Path.cwd(), *Path.cwd().parents]
root = next(p for p in roots if (p / 'configs/assets.json').exists())
base = root / 'results/runs/feature_audit_20261016_v1'
summary = json.loads((base / 'summary.json').read_text())
profile = pd.read_csv(base / 'feature_profile.csv')
metadata = pd.read_csv(base / 'metadata_by_ID.csv', keep_default_na=False, dtype={'id':str})
print({key:summary[key] for key in ['release','IDs_audited','rows_audited','fields_audited','data_modified','imputation_performed']})

{'release': 'per-drifter-20261016-v1', 'IDs_audited': 3741, 'rows_audited': 39107061, 'fields_audited': 130, 'data_modified': False, 'imputation_performed': False}


## Data

### Raw missingness in the pilot

The denominator is 318,108 observed development R origins. This table does not measure valid 7/30/365-day pairs.

In [2]:
pilot = profile[profile.segment.eq('development_R_origins')].set_index('feature')
cols = ['time','lat','lon','ve','vn','sst','sst1','sst2','err_ve','err_vn','err_lat','err_lon']
print(pilot.loc[cols,['null_rows','null_pct']].to_string())

         null_rows  null_pct
feature                     
time             0  0.000000
lat              0  0.000000
lon              0  0.000000
ve               0  0.000000
vn               0  0.000000
sst          15403  4.842066
sst1         15403  4.842066
sst2         15403  4.842066
err_ve       21801  6.853333
err_vn       21805  6.854590
err_lat        338  0.106253
err_lon        344  0.108139


## Results

### Temperature attrition and seasonal coverage

QC-unavailable includes raw missing values; do not add the two counts.

In [3]:
r = summary['segments']['development_R_origins']
for field in ['sst_ok','sst1_ok','sst2_ok']:
    unavailable = r['rows'] - r[field+'_true']
    print(field, unavailable, round(100*unavailable/r['rows'],3))
context = pd.read_csv(base/'context_counts.csv')
season = context[context.segment.eq('development_R_origins') & context.stratum.isin(['0','1','2','3'])].sort_values('stratum')
print(season[['stratum','rows','IDs','sst_ok_true','sst2_ok_true']].to_string(index=False))

sst_ok 15485 4.868
sst1_ok 15423 4.848
sst2_ok 92183 28.979
stratum    rows   IDs  sst_ok_true  sst2_ok_true
      0 75855.0 153.0      71069.0       48537.0
      1 55124.0 128.0      52132.0       38400.0
      2 87891.0 162.0      83584.0       67766.0
      3 99238.0 182.0      95838.0       71222.0


### ID-level metadata and independent reconciliation

Missing voltage is a source fill; zero depth has not been converted to 15 m. Independent ID counts by season must not be summed.

In [4]:
development = metadata[metadata.pool.eq('train')]
pilot_ids = development[development.R_visiting_ID]
print('Pilot depth values:',pilot_ids.DrogueCenterDepth.value_counts(dropna=False).to_dict())
print('ManufactureVoltage values:',development.ManufactureVoltage.value_counts(dropna=False).to_dict())
check=json.loads((base/'independent_checks.json').read_text())
assert check['passed']
assert check['result']['R_rows']==318108
assert check['result']['R_sst_unusable']==15485
print('Independent development check passed:',check['result'])

Pilot depth values: {'15.0': 376, '0.0': 20}
ManufactureVoltage values: {-1: 3224}
Independent development check passed: {'n_rows': 33943292.0, 'IDs': 3224.0, 'core_null_rows': 0.0, 'sst_missing': 1194141.0, 'sst_unusable': 1205537.0, 'sst2_unusable': 8361043.0, 'sst_missing_mask_disagreement': 0.0, 'negative_lat_uncertainty': 1342.0, 'absent_grid_hours': 620388.0, 'R_rows': 318108.0, 'R_sst_unusable': 15485.0, 'R_sst2_unusable': 92183.0}


## Takeaways

Preserve real position pairs without requiring SST. Review uncertainty, geometry and extreme motion separately. Keep structural nulls and unresolved hardware meanings explicit. Any optional imputation must be a separate, development-validated feature transformation with provenance; it must not manufacture observed targets or continuous paths. Annual/seasonal prediction accuracy is not established by this audit.